In [85]:
import pandas as pd
import numpy as np
import time
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [86]:
import importlib
import feature_engineer
importlib.reload(feature_engineer)

<module 'feature_engineer' from '/Users/jiashenwang/Desktop/Lucas_Systems_Capstone_Project/Model_Jiashen/feature_engineer.py'>

In [96]:
# Configurations
warehouses = ["OE", "OF"]
work_codes = ["10", "20", "30"]
max_time = 300
models = []
results = []

In [97]:
from feature_engineer import get_engineered_df

for wh in warehouses:
    DATA_PATH = f"../data/processed/{wh.lower()}_detailed.parquet"
    for wc in work_codes:
        print(f"--- Training Model for {wh} | WC {wc} ---")
        
        # 1. Preprocess
        df, features, cat_cols = get_engineered_df(DATA_PATH, warehouse=wh, work_code=wc, max_time=max_time)
        # remove distance from features
        # features = [f for f in features if f != "Travel_Distance"]
        
        # 2. Prepare Data (One-Hot Encoding)
        X = pd.get_dummies(df[features], columns=cat_cols, drop_first=True)
        y = df["Time_Delta_sec"]
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # 3. Model Training & Timing
        start_time = time.time()
        model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        end_time = time.time()
        models.append(model)
        
        # 4. Evaluation
        preds = model.predict(X_test)
        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        runtime = end_time - start_time
        
        # 5. Store Results
        results.append({
            "Warehouse": wh,
            "WorkCode": wc,
            "Train_Rows": len(X_train),
            "R^2": round(r2, 4),
            "MAE": round(mae, 2),
            "Runtime_Sec": round(runtime, 2)
        })

# Display Clean Results Table
results_df = pd.DataFrame(results)
display(results_df)

--- Training Model for OE | WC 10 ---
--- Training Model for OE | WC 20 ---
--- Training Model for OE | WC 30 ---
--- Training Model for OF | WC 10 ---
--- Training Model for OF | WC 20 ---
--- Training Model for OF | WC 30 ---


,Warehouse,WorkCode,Train_Rows,R^2,MAE,Runtime_Sec
0,OE,10,3265,0.2822,36.07,0.45
1,OE,20,17067,0.6619,16.38,0.56
2,OE,30,52238,0.3936,22.10,0.98
3,OF,10,3740,0.4209,30.50,0.33
4,OF,20,32110,0.6374,16.89,0.93
5,OF,30,60366,0.4046,27.80,1.24


In [91]:
models_nodist = []
results_nodist = []

for wh in warehouses:
    DATA_PATH = f"../data/processed/{wh.lower()}_detailed.parquet"
    for wc in work_codes:
        print(f"--- Training Model for {wh} | WC {wc} ---")
        
        # 1. Preprocess
        df, features, cat_cols = get_engineered_df(DATA_PATH, warehouse=wh, work_code=wc, max_time=max_time)
        # remove distance, "same_aisle", "same_lockey", "diff_level" from features
        features = [f for f in features if f not in ["Travel_Distance", "same_aisle", "same_lockey", "diff_level"]]
        cat_cols = [col for col in cat_cols if col not in ["same_aisle", "same_lockey", "diff_level"]]
        
        # 2. Prepare Data (One-Hot Encoding)
        X = pd.get_dummies(df[features], columns=cat_cols, drop_first=True)
        y = df["Time_Delta_sec"]
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # 3. Model Training & Timing
        start_time = time.time()
        model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        end_time = time.time()
        models_nodist.append(model)
        
        # 4. Evaluation
        preds = model.predict(X_test)
        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        runtime = end_time - start_time
        
        # 5. Store Results
        results_nodist.append({
            "Warehouse": wh,
            "WorkCode": wc,
            "Train_Rows": len(X_train),
            "R^2": round(r2, 4),
            "MAE": round(mae, 2),
            "Runtime_Sec": round(runtime, 2)
        })

# Display Clean Results Table
results_df_nodist = pd.DataFrame(results_nodist)
display(results_df_nodist)

--- Training Model for OE | WC 10 ---
--- Training Model for OE | WC 20 ---
--- Training Model for OE | WC 30 ---
--- Training Model for OF | WC 10 ---
--- Training Model for OF | WC 20 ---
--- Training Model for OF | WC 30 ---


,Warehouse,WorkCode,Train_Rows,R^2,MAE,Runtime_Sec
0,OE,10,3265,-0.1264,46.65,0.65
1,OE,20,17067,-0.0371,40.36,0.53
2,OE,30,52238,0.1214,27.22,0.97
3,OF,10,3740,-0.0502,48.21,0.30
4,OF,20,32110,0.1139,36.23,0.62
5,OF,30,60366,0.1252,34.37,0.86


In [101]:
# Append MAE column of nodist to original results_df, right after MAE column and befoire Runtime_Sec, also add percentage increase
display_df = results_df.copy()
display_df["MAE_noDist"] = results_df_nodist["MAE"]
display_df["MAE_Increase(%)"] = ((display_df["MAE_noDist"] - display_df["MAE"]) / display_df["MAE"] * 100).round(2)
# move Runtime_Sec column to the end
cols = list(display_df.columns)
cols.append(cols.pop(cols.index("Runtime_Sec")))
display_df = display_df[cols]
display(display_df) 

,Warehouse,WorkCode,Train_Rows,R^2,MAE,MAE_noDist,MAE_Increase(%),Runtime_Sec
0,OE,10,3265,0.2822,36.07,46.65,29.33,0.45
1,OE,20,17067,0.6619,16.38,40.36,146.40,0.56
2,OE,30,52238,0.3936,22.10,27.22,23.17,0.98
3,OF,10,3740,0.4209,30.50,48.21,58.07,0.33
4,OF,20,32110,0.6374,16.89,36.23,114.51,0.93
5,OF,30,60366,0.4046,27.80,34.37,23.63,1.24
